In [3]:
import sys
!{sys.executable} -m pip install --quiet --upgrade pip


In [4]:
import os
import json
import shutil

In [13]:
# === CONFIG ===
root_folder = r"C:\Users\shubo\Downloads\results_processed\results"
output_folder = os.path.join(root_folder, "valid_structures")
os.makedirs(output_folder, exist_ok=True)

In [14]:
# === Filter function ===
def is_valid_structure(metrics):
    try:
        return (
            metrics.get("avg_structure_validity", {}).get("value", 0) == 1.0 and
            metrics.get("avg_comp_validity", {}).get("value", 0) > 0.0 and
            metrics.get("avg_energy_above_hull_per_atom", {}).get("value", 1) < 0.1 and
            metrics.get("frac_successful_jobs", {}).get("value", 0) == 1.0 and
            metrics.get("frac_novel_structures", {}).get("value", 0) == 1.0 and
            metrics.get("frac_unique_structures", {}).get("value", 0) == 1.0
        )
    except Exception as e:
        print(f"❌ Error validating structure: {e}")
        return False

In [15]:
# === Process ===
valid_folders = []

for dir_name in os.listdir(root_folder):
    subdir = os.path.join(root_folder, dir_name)
    metrics_file = os.path.join(subdir, "metrics.json")
    
    if os.path.isdir(subdir) and os.path.isfile(metrics_file):
        try:
            with open(metrics_file, "r") as f:
                data = json.load(f)
            
            if is_valid_structure(data):
                valid_folders.append(dir_name)
                # Copy entire folder to output
                shutil.copytree(subdir, os.path.join(output_folder, dir_name))
        except Exception as e:
            print(f"⚠️ Failed to process {dir_name}: {e}")


In [16]:
# === Output ===
print("\n✅ Valid structures:")
for f in valid_folders:
    print(" •", f)

print(f"\nTotal valid structures: {len(valid_folders)}")
print(f"Copied to: {output_folder}")


✅ Valid structures:
 • molecule-10
 • molecule-1002
 • molecule-1006
 • molecule-1007
 • molecule-101
 • molecule-1010
 • molecule-1015
 • molecule-1016
 • molecule-1024
 • molecule-1025
 • molecule-103
 • molecule-1030
 • molecule-1031
 • molecule-1035
 • molecule-1037
 • molecule-1040
 • molecule-1042
 • molecule-1047
 • molecule-1051
 • molecule-1054
 • molecule-1060
 • molecule-1062
 • molecule-1066
 • molecule-1070
 • molecule-1075
 • molecule-1081
 • molecule-1082
 • molecule-1083
 • molecule-1085
 • molecule-1092
 • molecule-1094
 • molecule-1099
 • molecule-1100
 • molecule-1102
 • molecule-1103
 • molecule-1104
 • molecule-1107
 • molecule-1109
 • molecule-111
 • molecule-1115
 • molecule-1119
 • molecule-112
 • molecule-1126
 • molecule-1127
 • molecule-1128
 • molecule-1136
 • molecule-1147
 • molecule-1148
 • molecule-1150
 • molecule-1153
 • molecule-1154
 • molecule-1155
 • molecule-116
 • molecule-1161
 • molecule-1164
 • molecule-1167
 • molecule-1168
 • molecule-1175


In [20]:
import os
import json
import shutil
from pymatgen.core import Structure, Element
from pymatgen.core.periodic_table import DummySpecie

# === CONFIG ===
root_folder = r"C:\Users\shubo\Downloads\results_processed\results"
output_folder = os.path.join(root_folder, "valid_structures_no_heavy")
os.makedirs(output_folder, exist_ok=True)

# === Function to check for heavy elements ===
def contains_heavy_elements(structure_path):
    try:
        structure = Structure.from_file(structure_path)
        for site in structure:
            el = site.specie
            if isinstance(el, DummySpecie):
                continue
            Z = el.Z
            if (Z > 70 and el.symbol not in ["Pt", "Pb", "Bi"]) or el.symbol == "Tc":
                return True
        return False
    except Exception as e:
        print(f"⚠️ Failed to parse structure: {structure_path} ({e})")
        return True  # treat unparseable structures as invalid

# === Original MatterSim metric-based filter ===
def is_valid_structure(metrics):
    try:
        return (
            metrics.get("avg_structure_validity", {}).get("value", 0) == 1.0 and
            metrics.get("avg_comp_validity", {}).get("value", 0) > 0.0 and
            metrics.get("avg_energy_above_hull_per_atom", {}).get("value", 1) < 0.1 and
            metrics.get("frac_successful_jobs", {}).get("value", 0) == 1.0 and
            metrics.get("frac_novel_structures", {}).get("value", 0) == 1.0 and
            metrics.get("frac_unique_structures", {}).get("value", 0) == 1.0
        )
    except Exception as e:
        print(f"❌ Error validating structure: {e}")
        return False

# === MAIN PROCESS ===
valid_clean_folders = []

for dir_name in os.listdir(root_folder):
    subdir = os.path.join(root_folder, dir_name)
    metrics_file = os.path.join(subdir, "metrics.json")
    
    # Try common structure files
    structure_file = None
    for try_file in ["structure.json", "POSCAR", "structure.cif"]:
        try_path = os.path.join(subdir, try_file)
        if os.path.exists(try_path):
            structure_file = try_path
            break

    if os.path.isdir(subdir) and os.path.isfile(metrics_file) and structure_file:
        try:
            with open(metrics_file, "r") as f:
                metrics = json.load(f)

            if is_valid_structure(metrics) and not contains_heavy_elements(structure_file):
                valid_clean_folders.append(dir_name)
                shutil.copytree(subdir, os.path.join(output_folder, dir_name))
        except Exception as e:
            print(f"⚠️ Failed processing {dir_name}: {e}")

# === OUTPUT ===
print("\n✅ Valid structures WITHOUT heavy elements:")
for f in valid_clean_folders:
    print(" •", f)

print(f"\nTotal valid & clean structures: {len(valid_clean_folders)}")
print(f"Saved to: {output_folder}")



✅ Valid structures WITHOUT heavy elements:

Total valid & clean structures: 0
Saved to: C:\Users\shubo\Downloads\results_processed\results\valid_structures_no_heavy


In [22]:
import os
import shutil
from pymatgen.core import Structure

# === CONFIG ===
valid_folder = r"C:\Users\shubo\Downloads\results_processed\results\valid_structures"
li_folder = valid_folder + "_with_li"
os.makedirs(li_folder, exist_ok=True)

li_folders = []

for folder in os.listdir(valid_folder):
    folder_path = os.path.join(valid_folder, folder)

    # Assume structure file is gen.cif or gen.extxyz
    structure_file = None
    for try_file in ["gen.cif", "gen.extxyz"]:
        try_path = os.path.join(folder_path, try_file)
        if os.path.exists(try_path):
            structure_file = try_path
            break

    if structure_file:
        try:
            if structure_file.endswith(".cif"):
                structure = Structure.from_file(structure_file)
                elements = set(site.specie.symbol for site in structure)
                if "Li" in elements:
                    li_folders.append(folder)
                    shutil.copytree(folder_path, os.path.join(li_folder, folder))
            elif structure_file.endswith(".extxyz"):
                print(f"⚠️ Skipping {folder}: .extxyz support not yet added")
        except Exception as e:
            print(f"⚠️ Failed to process {folder}: {e}")

# === OUTPUT ===
print("\n✅ Final Li-containing structures (from gen.cif):")
for f in li_folders:
    print(" •", f)

print(f"\nTotal structures containing Li: {len(li_folders)}")
print(f"Copied to: {li_folder}")


C:\Users\shubo\anaconda3\Lib\site-packages\pymatgen\core\structure.py:3107: UserWarning: Issues encountered while parsing CIF: 1 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]



✅ Final Li-containing structures (from gen.cif):
 • molecule-1010
 • molecule-1031
 • molecule-1035
 • molecule-1037
 • molecule-1042
 • molecule-1070
 • molecule-1082
 • molecule-1085
 • molecule-1119
 • molecule-1127
 • molecule-1179
 • molecule-1246
 • molecule-1290
 • molecule-1304
 • molecule-1318
 • molecule-1319
 • molecule-1327
 • molecule-1328
 • molecule-1375
 • molecule-140
 • molecule-1406
 • molecule-1436
 • molecule-147
 • molecule-1539
 • molecule-1559
 • molecule-1579
 • molecule-16
 • molecule-1612
 • molecule-1685
 • molecule-1686
 • molecule-1699
 • molecule-1737
 • molecule-1747
 • molecule-1774
 • molecule-1817
 • molecule-1829
 • molecule-1843
 • molecule-196
 • molecule-1970
 • molecule-1981
 • molecule-199
 • molecule-2005
 • molecule-201
 • molecule-2083
 • molecule-2093
 • molecule-2101
 • molecule-2124
 • molecule-2202
 • molecule-2209
 • molecule-222
 • molecule-2226
 • molecule-2254
 • molecule-2259
 • molecule-228
 • molecule-2286
 • molecule-2289
 • mole

In [23]:
import os
import shutil
from pymatgen.core import Structure, Element
from pymatgen.core.periodic_table import DummySpecie

# === CONFIG ===
li_folder = r"C:\Users\shubo\Downloads\results_processed\results\valid_structures_with_li"
clean_folder = li_folder + "_no_heavy"
os.makedirs(clean_folder, exist_ok=True)

clean_folders = []

for folder in os.listdir(li_folder):
    folder_path = os.path.join(li_folder, folder)

    # Look for gen.cif only (assumed structure file)
    structure_file = os.path.join(folder_path, "gen.cif")
    if not os.path.exists(structure_file):
        print(f"❌ No gen.cif in {folder}")
        continue

    try:
        structure = Structure.from_file(structure_file)
        has_heavy = False
        for site in structure:
            el = site.specie
            if isinstance(el, DummySpecie):
                continue
            Z = el.Z
            if (Z > 70 and el.symbol not in ["Pt", "Pb", "Bi"]) or el.symbol == "Tc":
                has_heavy = True
                break

        if not has_heavy:
            clean_folders.append(folder)
            shutil.copytree(folder_path, os.path.join(clean_folder, folder))
    except Exception as e:
        print(f"⚠️ Error processing {folder}: {e}")

# === OUTPUT ===
print("\n✅ Li-containing structures WITHOUT heavy elements:")
for f in clean_folders:
    print(" •", f)

print(f"\nFinal total: {len(clean_folders)}")
print(f"Saved to: {clean_folder}")



✅ Li-containing structures WITHOUT heavy elements:
 • molecule-1010
 • molecule-1031
 • molecule-1035
 • molecule-1042
 • molecule-1070
 • molecule-1082
 • molecule-1085
 • molecule-1127
 • molecule-1246
 • molecule-1290
 • molecule-1304
 • molecule-1318
 • molecule-1319
 • molecule-1327
 • molecule-147
 • molecule-1579
 • molecule-16
 • molecule-1612
 • molecule-1685
 • molecule-1686
 • molecule-1699
 • molecule-1774
 • molecule-1817
 • molecule-1829
 • molecule-1843
 • molecule-1970
 • molecule-199
 • molecule-201
 • molecule-2083
 • molecule-2093
 • molecule-2101
 • molecule-2202
 • molecule-2209
 • molecule-222
 • molecule-2226
 • molecule-2254
 • molecule-2259
 • molecule-228
 • molecule-2286
 • molecule-2289
 • molecule-2296
 • molecule-2376
 • molecule-244
 • molecule-2443
 • molecule-2455
 • molecule-2460
 • molecule-2510
 • molecule-2533
 • molecule-254
 • molecule-2595
 • molecule-2630
 • molecule-264
 • molecule-2647
 • molecule-2666
 • molecule-2668
 • molecule-269
 • mole